# CSA — seed + verify all 8 scenarios (S16.3 close-out)

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 16.5 mini-sprint — S16.3 close-out via **Path A** (Fabric notebook + MPE) |
| **Why not GH runner?** | Enterprise policy `CosmosDB_PublicNetwork_Modify` at Tenant Root Group scope enforces `publicNetworkAccess: Disabled` on every Cosmos DB account. Every PATCH is silently reverted, so a GH-hosted runner in the SIT env can never reach `cosmos-csa-ihzhhpf-sit`. Long-term CI path (self-hosted runner in VNet) is tracked as a Sprint 17 issue (see repo issues) but scoped out for the MCAPS demo tenant. |
| **Cosmos target** | `cosmos-csa-ihzhhpf-sit` / `csa` / `scenarios` (partition key `/scenarioId`) — private endpoint via Fabric MPE `mpe-cosmos-csa-ihzhhpf-sit` |
| **Auth** | `notebookutils.credentials.getToken('https://cosmos.azure.com')` (same pattern as `csa-verify-mvp` — `DefaultAzureCredential` isn't supported in Fabric Spark) |
| **Result** | On success: 8 scenarios upserted + assert `count == 8` passes. Fabric job status `Completed` = proof that all 8 are in Cosmos. |

Source of truth for the scenario documents remains `data/csa/scenarios/*.yaml`
in the repo — this notebook is a generated artefact built by
`.scratch/publish_and_run_seed_scenarios.py`; do not hand-edit the seed
cell. Re-run the authoring script whenever the YAML catalogue changes.


In [ ]:
# Fabric-native Cosmos client — mirrors csa-verify-mvp Cell 1.
# `azure-cosmos` is provided by the env-csa environment attached in
# notebook metadata. `notebookutils` is auto-injected by Fabric.
import time as _time
from azure.core.credentials import AccessToken
from azure.cosmos import CosmosClient

COSMOS_URL = 'https://cosmos-csa-ihzhhpf-sit.documents.azure.com:443/'
DB_NAME = 'csa'
CONTAINER_NAME = 'scenarios'


class FabricNBUToken:
    """Adapter that returns Fabric-brokered tokens for the Cosmos audience."""

    def get_token(self, *_scopes, **_kwargs):
        raw = notebookutils.credentials.getToken('https://cosmos.azure.com')
        # Fabric tokens are ~60 min; cache for 55.
        return AccessToken(raw, int(_time.time()) + 3300)


print('Connecting to Cosmos via Fabric MPE...')
cred = FabricNBUToken()
client = CosmosClient(COSMOS_URL, credential=cred)
db = client.get_database_client(DB_NAME)
container = db.get_container_client(CONTAINER_NAME)
print(f'  target: {DB_NAME}/{CONTAINER_NAME}')


In [ ]:
# Seed all 8 scenarios into Cosmos via MPE. Source of truth remains
# `data/csa/scenarios/*.yaml` — this cell is generated from those files
# by `.scratch/publish_and_run_seed_scenarios.py` (do not hand-edit).

SCENARIOS = [ { 'affectedResources': ['burn-beds', 'icu-beds', 'or-slots'],
    'cascade': ['burn-icu-overwhelm', 'inter-hospital-transfers'],
    'defaultTier': 3,
    'duration': {'unit': 'days', 'value': 5},
    'family': 'F3',
    'hospitalRelevance': 'Special capability (burn ICU) overwhelmed — multi-site coordination '
                         '(Ausserordentliche Lage).',
    'id': 'crans-montana-burns-mci',
    'kpis': ['burn-bed-shortfall', 'inter-hospital-transfers-count'],
    'magnitude': {'unit': 'percent', 'value': 200},
    'mvpRequired': False,
    'name': 'Crans-Montana burns MCI',
    'onset': 'sudden',
    'responseLevers': [ 'lever-activate-burn-surge-protocol',
                        'lever-stand-up-mass-casualty-reception-zone',
                        'lever-request-rega-air-transfer-coordination',
                        'lever-escalate-to-cantonal-medical-command'],
    'scarceCapability': 'burn-beds',
    'scenarioId': 'crans-montana-burns-mci',
    'shockVector': 'demand-surge',
    'trigger': 'A mass-casualty incident produces a surge of severe burns requiring specialist '
               'burn and ICU capacity.'},
  { 'affectedResources': ['icu-beds', 'beds', 'it-systems'],
    'cascade': ['manual-workarounds', 'throughput-collapse', 'diagnostics-degraded'],
    'defaultTier': 3,
    'duration': {'unit': 'days', 'value': 4},
    'family': 'F4',
    'hospitalRelevance': 'Systemic IT loss cascading to capacity — escalates beyond internal '
                         'levers (Tier 2-3).',
    'id': 'cyberattack-hospital-services',
    'kpis': ['throughput-reduction-pct', 'icu-bed-shortfall'],
    'magnitude': {'unit': 'percent', 'value': 30},
    'mvpRequired': True,
    'name': 'Cyberattack on hospital services',
    'onset': 'sudden',
    'responseLevers': [ 'lever-fail-over-to-backup-clinical-it-systems',
                        'lever-activate-downtime-paper-procedures',
                        'lever-isolate-affected-network-segments',
                        'lever-engage-cyber-incident-response-retainer',
                        'lever-protect-critical-care-from-it-outage-impact'],
    'scarceCapability': 'it-systems',
    'scenarioId': 'cyberattack-hospital-services',
    'shockVector': 'capacity-loss',
    'trigger': 'Ransomware disables clinical IT, degrading throughput across ED, wards, and '
               'critical care.'},
  { 'affectedResources': ['ed-capacity', 'or-slots'],
    'cascade': ['delayed-critical-transfers', 'ed-crowding'],
    'defaultTier': 2,
    'duration': {'unit': 'hours', 'value': 8},
    'family': 'F1',
    'hospitalRelevance': 'Single-site logistics failure requiring internal reallocation and EMS '
                         'diversion.',
    'id': 'helipad-elevator-failure',
    'kpis': ['critical-transfer-delay-minutes', 'ed-occupancy-pct'],
    'magnitude': {'unit': 'percent', 'value': 25},
    'mvpRequired': False,
    'name': 'Helipad elevator failure',
    'onset': 'sudden',
    'responseLevers': [ 'lever-activate-hospital-emergency-operations-centre',
                        'lever-divert-ambulances-to-partner-sites',
                        'lever-prioritise-or-by-clinical-urgency'],
    'scarceCapability': None,
    'scenarioId': 'helipad-elevator-failure',
    'shockVector': 'capacity-loss',
    'trigger': 'Rooftop helipad elevator fails, blocking rapid transfer of critical arrivals to '
               'the ED/OR.'},
  { 'affectedResources': ['pediatric-beds', 'nursing-staff'],
    'cascade': ['pediatric-bed-pressure', 'staffing-strain'],
    'defaultTier': 2,
    'duration': {'unit': 'weeks', 'value': 4},
    'family': 'F6',
    'hospitalRelevance': 'Canonical single-site seasonal surge managed by internal reallocation '
                         '(Besondere Lage).',
    'id': 'pediatric-virus-surge-rsv',
    'kpis': ['pediatric-bed-shortfall', 'pediatric-occupancy-pct'],
    'magnitude': {'unit': 'percent', 'value': 50},
    'mvpRequired': True,
    'name': 'Pediatric virus surge (RSV)',
    'onset': 'gradual',
    'responseLevers': [ 'lever-open-pediatric-overflow-cohort',
                        'lever-recall-off-duty-clinical-staff',
                        'lever-accelerate-discharge-of-medically-fit-patients',
                        'lever-defer-non-urgent-elective-admissions'],
    'scarceCapability': 'pediatric-beds',
    'scenarioId': 'pediatric-virus-surge-rsv',
    'shockVector': 'demand-surge',
    'trigger': 'A seasonal RSV wave drives a surge in pediatric admissions, pressuring pediatric '
               'beds and staff.'},
  { 'affectedResources': ['beds', 'ed-capacity'],
    'cascade': ['bed-pressure', 'ed-crowding'],
    'defaultTier': 2,
    'duration': {'unit': 'days', 'value': 7},
    'family': 'F8',
    'hospitalRelevance': 'Bed-pressure heatwave managed by flow levers and early discharge '
                         '(Besondere Lage).',
    'id': 'summer-heatwave-demand-surge',
    'kpis': ['bed-shortfall', 'ed-occupancy-pct'],
    'magnitude': {'unit': 'percent', 'value': 20},
    'mvpRequired': True,
    'name': 'Summer heatwave demand surge',
    'onset': 'gradual',
    'responseLevers': [ 'lever-accelerate-discharge-of-medically-fit-patients',
                        'lever-activate-discharge-lounge',
                        'lever-mobilise-spitex-for-discharge-support',
                        'lever-redirect-walk-ins-to-urgent-care-partners'],
    'scarceCapability': None,
    'scenarioId': 'summer-heatwave-demand-surge',
    'shockVector': 'demand-surge',
    'trigger': 'A prolonged heatwave drives a bed-pressure surge in elderly and cardiorespiratory '
               'admissions.'},
  { 'affectedResources': ['ed-capacity', 'or-slots', 'icu-beds', 'blood'],
    'cascade': ['mass-casualty-reception', 'security-lockdown', 'blood-demand-spike'],
    'defaultTier': 3,
    'duration': {'unit': 'days', 'value': 3},
    'family': 'F7',
    'hospitalRelevance': 'Severe-consequence multi-agency event with second-hit risk '
                         '(Ausserordentliche Lage).',
    'id': 'terror-attack-second-hit-risk',
    'kpis': ['casualty-throughput', 'blood-units-consumed'],
    'magnitude': {'unit': 'percent', 'value': 150},
    'mvpRequired': False,
    'name': 'Terror attack with risk of second hit',
    'onset': 'sudden',
    'responseLevers': [ 'lever-stand-up-mass-casualty-reception-zone',
                        'lever-prioritise-blood-product-allocation',
                        'lever-escalate-to-cantonal-medical-command',
                        'lever-activate-media-liaison-protocol'],
    'scarceCapability': 'icu-beds',
    'scenarioId': 'terror-attack-second-hit-risk',
    'shockVector': 'demand-surge',
    'trigger': 'A terror attack produces a casualty surge while a credible risk of a second strike '
               'constrains response.'},
  { 'affectedResources': ['ventilators'],
    'cascade': ['ventilated-capacity-loss', 'inter-site-reallocation'],
    'defaultTier': 3,
    'duration': {'unit': 'weeks', 'value': 2},
    'family': 'F5',
    'hospitalRelevance': 'Scarce special capability (ventilators) — cantonal stockpile and '
                         'reallocation required.',
    'id': 'ventilator-supply-shortage',
    'kpis': ['ventilator-shortfall', 'ventilated-patient-days-at-risk'],
    'magnitude': {'unit': 'percent', 'value': 40},
    'mvpRequired': False,
    'name': 'Ventilator supply shortage',
    'onset': 'rapid',
    'responseLevers': [ 'lever-draw-down-ventilator-reserve-stock',
                        'lever-request-cantonal-stockpile-release',
                        'lever-reallocate-ventilators-across-sites',
                        'lever-switch-to-alternate-substitute-devices'],
    'scarceCapability': 'ventilators',
    'scenarioId': 'ventilator-supply-shortage',
    'shockVector': 'supply-loss',
    'trigger': 'A supply-chain disruption depletes the ventilator reserve during elevated '
               'respiratory demand.'},
  { 'affectedResources': ['physician-staff', 'nursing-staff'],
    'cascade': ['slower-decisions', 'discharge-delays'],
    'defaultTier': 2,
    'duration': {'unit': 'days', 'value': 3},
    'family': 'F2',
    'hospitalRelevance': 'Planned staffing dip manageable with reinforced internal means '
                         '(Besondere Lage).',
    'id': 'ward-specialists-at-congress',
    'kpis': ['senior-cover-ratio', 'discharge-delay-hours'],
    'magnitude': {'unit': 'percent', 'value': 30},
    'mvpRequired': False,
    'name': 'Ward specialists at congress',
    'onset': 'gradual',
    'responseLevers': [ 'lever-recall-off-duty-clinical-staff',
                        'lever-backfill-specialists-remotely-via-telemedicine',
                        'lever-cross-deploy-staff-from-elective-areas'],
    'scarceCapability': None,
    'scenarioId': 'ward-specialists-at-congress',
    'shockVector': 'staff-loss',
    'trigger': 'A cohort of ward specialists is simultaneously absent at a medical congress, '
               'thinning senior cover.'}]

print(f'Upserting {len(SCENARIOS)} scenarios into {DB_NAME}/{CONTAINER_NAME} via MPE...')
_upserted = 0
for _doc in SCENARIOS:
    container.upsert_item(_doc)
    _upserted += 1
    print(f'  ok  {_doc["scenarioId"]}')
print(f'\nUpserted {_upserted}/{len(SCENARIOS)} scenarios.')


In [ ]:
# Verify: full scan, count, assert == 8, print sorted summary.
_items = list(
    container.query_items(
        query='SELECT c.id, c.scenarioId, c.name, c.family, c.defaultTier, c.mvpRequired FROM c',
        enable_cross_partition_query=True,
    )
)
print(f'\nscenarios in container: {len(_items)}')
for _s in sorted(_items, key=lambda x: str(x.get('scenarioId', ''))):
    _mvp = ' [MVP]' if _s.get('mvpRequired') else ''
    print(f"  {_s.get('scenarioId', '?')}  family={_s.get('family', '?')}  tier={_s.get('defaultTier', '?')}  {_s.get('name', '?')}{_mvp}")

if len(_items) != 8:
    raise AssertionError(
        f'S16.3 verification FAILED: expected 8 scenarios in Cosmos, got {len(_items)}. '
        'Re-run the authoring script and retry.'
    )
print('\nS16.3 verification PASSED: all 8 seeded scenarios present in Cosmos.')
